# Embeddings & Vector Search

Photos stored in S3 and indexed in Postgres give us exact lookups: retrieve photo 42, list photos from 2024, filter by camera model. What they cannot do is answer the question "show me photos that *feel* like a rainy afternoon in Tokyo." That kind of query requires a fundamentally different data structure: an **embedding index**.

In this notebook we compute **CLIP embeddings** for every photo and build the semantic search layer on top of pgvector. We cover the theory of multimodal embeddings, implement the full embedding pipeline, and expose search endpoints through the FastAPI router introduced in earlier notebooks.

## Embedding Theory

An **embedding** is a function $f\colon \mathcal{X} \to \mathbb{R}^d$ that maps raw inputs (images, text strings, audio clips) into a fixed-dimensional vector space such that distance in that space reflects semantic similarity. Formally, we want:

$$\text{sim}(f(x_1), f(x_2)) \text{ is high} \iff x_1 \text{ and } x_2 \text{ are semantically similar.}$$

The dimension $d$ is a design choice; common values are $128$, $512$, and $1536$. Smaller $d$ means faster search and less storage but lower expressivity. The specific geometry of $\mathbb{R}^d$ that matters most is **cosine similarity**:

$$\text{sim}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|\|\mathbf{v}\|}$$

Cosine similarity is scale-invariant: only the *direction* of the vector encodes meaning, not its magnitude. This makes it the preferred metric for text and vision embeddings, which are typically $\ell_2$-normalized before storage. The complementary **cosine distance** $1 - \text{sim}(\mathbf{u}, \mathbf{v}) \in [0, 2]$ is what pgvector minimizes when you write `ORDER BY embedding <=> query_vec`.

<br>

**Contrastive learning.** The embedding function $f$ must be learned. For **CLIP** (Contrastive Language–Image Pre-training), training proceeds on a batch of $N$ image-text pairs $(\mathbf{I}_i, \mathbf{T}_i)_{i=1}^N$ scraped from the web. The image encoder $f_I$ and text encoder $f_T$ each produce $\mathbb{R}^{512}$ vectors. The loss is the **InfoNCE loss**, which treats each pair as a positive and all $N^2 - N$ cross-pairs as negatives:

$$\mathcal{L} = -\frac{1}{2N} \sum_{i=1}^N \left[ \log \frac{\exp(f_I(\mathbf{I}_i) \cdot f_T(\mathbf{T}_i) / \tau)}{\sum_{j=1}^N \exp(f_I(\mathbf{I}_i) \cdot f_T(\mathbf{T}_j) / \tau)} + \log \frac{\exp(f_T(\mathbf{T}_i) \cdot f_I(\mathbf{I}_i) / \tau)}{\sum_{j=1}^N \exp(f_T(\mathbf{T}_i) \cdot f_I(\mathbf{I}_j) / \tau)} \right]$$

where $\tau$ is a learned temperature. Minimizing $\mathcal{L}$ pushes paired embeddings together and unrelated embeddings apart. The remarkable consequence is that after training, `f_I(photo_of_a_sunset)` and `f_T("sunset over the ocean")` end up close in the *same* $\mathbb{R}^{512}$ space, enabling cross-modal search.

:::{.callout-note}
The InfoNCE loss is a lower bound on mutual information between image and text representations. Larger batch sizes $N$ provide more negatives per step and tighten this bound, which is one reason CLIP was trained on batches of 32,768 pairs across 256 GPUs.

:::

## CLIP Architecture

CLIP is a **two-tower model**: an image encoder and a text encoder that each produce vectors in a shared $\mathbb{R}^{512}$ space.

**Image encoder.** We use **ViT-B/32**, a Vision Transformer with $32 \times 32$ pixel patches, 12 transformer layers, and 512-dim output. The input image is divided into a $7 \times 7$ grid of patches (for $224 \times 224$ input), each patch is linearly projected, and the sequence is processed by a standard Transformer encoder. The `[CLS]` token embedding after the last layer is projected to $\mathbb{R}^{512}$ and $\ell_2$-normalized.

**Text encoder.** A 12-layer Transformer operating on BPE tokens; the embedding of the `[EOS]` token is projected to $\mathbb{R}^{512}$ and $\ell_2$-normalized.

Both towers project to the *same* embedding space, so image and text embeddings are directly comparable with dot product or cosine similarity. This is the key property that enables text-to-image search without any image-text pair at query time — we just encode the query string and retrieve the nearest image embeddings.

The `openai/CLIP` Python package exposes `clip.load("ViT-B/32")` which returns a `(model, preprocess)` pair. `preprocess` is a `torchvision.transforms` pipeline: resize to $224 \times 224$, center-crop, convert to tensor, normalize with ImageNet statistics.

Loading the CLIP model and running a forward pass:

In [ ]:
import clip
import torch
from PIL import Image
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)
model.eval()
print(f"Model loaded on {device}")
print(f"Input resolution: {model.visual.input_resolution}")
print(f"Embedding dimension: {model.visual.output_dim}")

Encoding a synthetic image and a text prompt, then computing their cosine similarity:

In [ ]:
# Synthetic 224x224 RGB image (solid orange)
img = Image.fromarray(
    np.full((224, 224, 3), [255, 140, 0], dtype=np.uint8)
)

with torch.no_grad():
    image_input = preprocess(img).unsqueeze(0).to(device)
    text_input  = clip.tokenize(["a bright orange image"]).to(device)

    image_vec = model.encode_image(image_input)
    text_vec  = model.encode_text(text_input)

    # L2-normalize
    image_vec = image_vec / image_vec.norm(dim=-1, keepdim=True)
    text_vec  = text_vec  / text_vec.norm(dim=-1, keepdim=True)

    similarity = (image_vec @ text_vec.T).item()

print(f"Image embedding shape : {image_vec.shape}")
print(f"Text embedding shape  : {text_vec.shape}")
print(f"Cosine similarity     : {similarity:.4f}")

## Embedding Pipeline

The embedding pipeline runs on every photo ingested into the system. Its entry point is `embed_photo(image_bytes: bytes) -> np.ndarray`, which accepts raw bytes as downloaded from S3 and returns a normalized `float32` vector of shape $(512,)$ — the CLIP ViT-B/32 embedding dimension.

**Interface design.** The function takes bytes rather than a `PIL.Image` or a file path: S3 objects arrive as byte streams, and keeping the interface at the bytes level avoids unnecessary intermediate file I/O. The caller (the S3 pipeline worker from notebook 08) downloads the image once and passes the same bytes to both `extract_exif` and `embed_photo`, avoiding a second download.

**CPU vs. GPU.** On a modern GPU (A100), CLIP encodes a single image in under 1ms; on CPU, approximately 50–100ms. For the personal photo library at scale (50K photos), even CPU inference adds only ~1 hour of total embedding time — acceptable for a one-shot indexing job. Ongoing incremental indexing (new photos arrive daily) is fast enough on CPU that no GPU is required.

Defining the single-image embedding function:

In [ ]:
import io

def embed_photo(image_bytes: bytes) -> np.ndarray:
    """Encode raw image bytes into a normalized CLIP embedding."""
    img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    tensor = preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        vec = model.encode_image(tensor)
        vec = vec / vec.norm(dim=-1, keepdim=True)
    return vec.squeeze(0).cpu().numpy().astype(np.float32)

For ingestion we need to process $B$ images at once rather than calling `model.encode_image` in a tight loop. Batching amortizes the Python/CUDA launch overhead and exploits the matrix-level parallelism of the transformer — the speedup is roughly $B\times$ on CPU and larger on GPU.

Defining the batch embedding function and benchmarking against single-image calls:

In [ ]:
import time

def embed_batch(
    image_bytes_list: list[bytes],
    batch_size: int = 32,
) -> np.ndarray:
    """Encode a list of raw image bytes, returning shape (N, 512)."""
    all_vecs = []
    for i in range(0, len(image_bytes_list), batch_size):
        chunk = image_bytes_list[i : i + batch_size]
        imgs  = [Image.open(io.BytesIO(b)).convert("RGB") for b in chunk]
        tensors = torch.stack([preprocess(im) for im in imgs]).to(device)
        with torch.no_grad():
            vecs = model.encode_image(tensors)
            vecs = vecs / vecs.norm(dim=-1, keepdim=True)
        all_vecs.append(vecs.cpu().numpy().astype(np.float32))
    return np.concatenate(all_vecs, axis=0)


# --- benchmark: 3 synthetic images ---
def _make_bytes(color: tuple) -> bytes:
    buf = io.BytesIO()
    Image.fromarray(np.full((224, 224, 3), color, dtype=np.uint8)).save(buf, format="JPEG")
    return buf.getvalue()

images = [_make_bytes((r, g, b)) for r, g, b in [(255, 0, 0), (0, 255, 0), (0, 0, 255)]]

t0 = time.perf_counter()
singles = np.stack([embed_photo(b) for b in images])
t_single = time.perf_counter() - t0

t0 = time.perf_counter()
batched = embed_batch(images, batch_size=32)
t_batch = time.perf_counter() - t0

print(f"Single-by-single : {t_single*1000:.1f} ms")
print(f"Batched          : {t_batch*1000:.1f} ms")
print(f"Output shape     : {batched.shape}")
print(f"Max abs diff     : {np.abs(singles - batched).max():.2e}")

**Integration with the S3 pipeline.** In the photo ingestion worker (introduced in notebook 08), the embedding step slots in immediately after EXIF extraction. The worker calls `embed_photo(image_bytes)` and upserts the result into `photos.embedding`. Because `embed_photo` is CPU-bound, we run it in a thread pool via `asyncio.to_thread` to avoid blocking the async event loop:

```python
embedding = await asyncio.to_thread(embed_photo, image_bytes)
await session.execute(
    update(Photo)
    .where(Photo.id == photo_id)
    .values(embedding=embedding.tolist())
)
```

## pgvector Indexing

With embeddings stored in Postgres, we need an index that makes nearest-neighbor queries fast. pgvector supports two approximate index types: **IVFFlat** and **HNSW**. We use HNSW for production.

**IVFFlat** partitions the embedding space into `lists` Voronoi cells using k-means, then at query time searches only the nearest `probes` cells. Build is fast and memory usage is low, but recall can degrade when the query falls near a cell boundary.

**HNSW** (Hierarchical Navigable Small World) builds a multi-layer proximity graph. Each node at layer $\ell$ is connected to $m$ neighbors; queries traverse from the top layer downward, greedily following edges toward the query vector. Two parameters control the index:
- $m$: number of bidirectional links per node. Higher $m$ improves recall at the cost of more memory and slower build.
- $ef\_construction$: search width during index build. Higher values build a higher-quality graph at the cost of build time.

Query time is $O(\log N)$ for HNSW vs. $O(N / \text{lists})$ for IVFFlat. For a photo library of even a few million images, HNSW is the right default.

The DDL to add the column and create the index:

In [ ]:
from sqlalchemy import text

# Assumes `engine` is a SQLAlchemy async engine connected to Postgres with pgvector
MIGRATION_DDL = [
    "CREATE EXTENSION IF NOT EXISTS vector",
    "ALTER TABLE photos ADD COLUMN IF NOT EXISTS embedding vector(512)",
    """
    CREATE INDEX IF NOT EXISTS photos_embedding_hnsw_idx
    ON photos
    USING hnsw (embedding vector_cosine_ops)
    WITH (m = 16, ef_construction = 64)
    """,
]

# Dry-run: print the statements we would execute
for stmt in MIGRATION_DDL:
    print(stmt.strip())
    print()

1. `CREATE EXTENSION IF NOT EXISTS vector`: loads the pgvector extension. This must run once per database, typically in the initial migration.
2. `ALTER TABLE photos ADD COLUMN IF NOT EXISTS embedding vector(512)`: adds the column; the `vector(512)` type is provided by pgvector and stores a fixed-length float array of dimension $512$.
3. `USING hnsw (embedding vector_cosine_ops)`: specifies cosine distance as the metric; `vector_cosine_ops` tells the index to optimize for the `<=>` operator (cosine distance). Use `vector_l2_ops` for Euclidean distance (`<->`) if your embeddings are not $\ell_2$-normalized.

:::{.callout-caution}
HNSW indexes in pgvector are built *at index creation time* and do not auto-rebuild when rows are inserted. Rows inserted after the index is created are still searchable (pgvector appends them to the graph), but rebuilding the index periodically with `REINDEX` improves recall as the dataset grows.

:::

## Semantic Search API

We expose two search endpoints:

- `POST /search/`: text-to-image search. The body is `{"query": "...", "limit": 20}`. The handler encodes the query text with CLIP and retrieves the `limit` nearest photos by cosine distance.
- `GET /photos/{id}/similar?limit=10`: image-to-image search. The stored embedding of photo `{id}` is used as the query vector.

Both endpoints return a list of `PhotoRead` schemas (defined in notebook 08). The SQLAlchemy ORM pattern for the nearest-neighbor query is:

```python
select(Photo).order_by(Photo.embedding.cosine_distance(query_vec)).limit(k)
```

where `cosine_distance` is a method added to mapped columns by the `pgvector-sqlalchemy` integration (`from pgvector.sqlalchemy import Vector`).

Defining the request model and search router:

In [ ]:
from pydantic import BaseModel, Field
from typing import List

# --- Request / response schemas ---

class SearchRequest(BaseModel):
    query: str = Field(..., min_length=1, max_length=400, description="Natural-language search query")
    limit: int = Field(20, ge=1, le=100)


class PhotoRead(BaseModel):
    """Minimal read schema — real version mirrors the Photo ORM model."""
    id: int
    s3_key: str
    taken_at: str | None
    score: float | None = None  # cosine similarity (1 - distance)

    model_config = {"from_attributes": True}


print(SearchRequest.model_json_schema())

Defining the search router with both endpoints:

In [ ]:
# This cell shows the router code; it does not run standalone without a database.
SEARCH_ROUTER = '''
from fastapi import APIRouter, Depends, HTTPException
from sqlalchemy.ext.asyncio import AsyncSession
from sqlalchemy import select
import asyncio, torch, clip
import numpy as np

from .database import get_session   # yields AsyncSession
from .models   import Photo         # SQLAlchemy ORM model
from .schemas  import PhotoRead, SearchRequest
from .embed    import model as clip_model, preprocess, device

router = APIRouter(prefix="", tags=["search"])


def _encode_text(query: str) -> list[float]:
    tokens = clip.tokenize([query]).to(device)
    with torch.no_grad():
        vec = clip_model.encode_text(tokens)
        vec = vec / vec.norm(dim=-1, keepdim=True)
    return vec.squeeze(0).cpu().numpy().astype(float).tolist()


@router.post("/search/", response_model=list[PhotoRead])
async def search_photos(
    body: SearchRequest,
    session: AsyncSession = Depends(get_session),
):
    query_vec = await asyncio.to_thread(_encode_text, body.query)
    stmt = (
        select(Photo)
        .where(Photo.embedding.isnot(None))
        .order_by(Photo.embedding.cosine_distance(query_vec))
        .limit(body.limit)
    )
    rows = (await session.scalars(stmt)).all()
    return rows


@router.get("/photos/{photo_id}/similar", response_model=list[PhotoRead])
async def similar_photos(
    photo_id: int,
    limit: int = 10,
    session: AsyncSession = Depends(get_session),
):
    photo = await session.get(Photo, photo_id)
    if photo is None:
        raise HTTPException(status_code=404, detail="Photo not found")
    if photo.embedding is None:
        raise HTTPException(status_code=422, detail="Photo has no embedding")
    stmt = (
        select(Photo)
        .where(Photo.id != photo_id)
        .where(Photo.embedding.isnot(None))
        .order_by(Photo.embedding.cosine_distance(photo.embedding))
        .limit(limit)
    )
    rows = (await session.scalars(stmt)).all()
    return rows
'''
print(SEARCH_ROUTER.strip())

## Cost vs. Recall Tradeoffs

Approximate nearest-neighbor (ANN) search trades recall for speed. **Exact KNN** computes the distance from the query to every row and sorts; it is $O(N \cdot d)$ and always correct, but infeasible for large $N$. HNSW reduces query time to $O(\log N)$ at the cost of occasionally missing the true nearest neighbor.

**Recall at $k$** is the standard metric:

$$\text{Recall}@k = \frac{|\text{HNSW top-}k \cap \text{exact top-}k|}{k}$$

In pgvector, the `ef_search` parameter (set per session with `SET hnsw.ef_search = 100`) controls the search width at query time: higher values traverse more candidate nodes in the graph, improving recall at the cost of latency. The default is `40`; for photo search (low latency requirements, high recall desirable) `100`–`200` is a reasonable operating point.

The tradeoff can be measured with a simple benchmark: generate $Q$ random queries, retrieve top-$k$ results from both exact KNN and HNSW, compute the intersection.

Mock recall benchmark using a small in-memory embedding matrix:

In [ ]:
rng = np.random.default_rng(42)

N = 10_000   # number of stored embeddings
d = 512      # embedding dimension
Q = 100      # number of query vectors
k = 10       # top-k

# Normalized embeddings (simulates CLIP output stored in pgvector)
db = rng.standard_normal((N, d)).astype(np.float32)
db /= np.linalg.norm(db, axis=1, keepdims=True)

queries = rng.standard_normal((Q, d)).astype(np.float32)
queries /= np.linalg.norm(queries, axis=1, keepdims=True)

# --- Exact KNN (brute-force) ---
sims = queries @ db.T              # (Q, N) cosine similarities
exact_top_k = np.argsort(-sims, axis=1)[:, :k]   # (Q, k)

# --- Simulated HNSW: exact + 5% random noise (mimics recall ~0.95) ---
def simulate_hnsw(queries, db, k, drop_rate=0.05):
    sims = queries @ db.T
    top_k = np.argsort(-sims, axis=1)[:, :k]
    # randomly replace a fraction of results with a non-top-k index
    mask = rng.random(top_k.shape) < drop_rate
    top_k[mask] = rng.integers(k, N, size=mask.sum())
    return top_k

approx_top_k = simulate_hnsw(queries, db, k, drop_rate=0.05)

# --- Recall computation ---
recalls = []
for q in range(Q):
    intersection = len(set(exact_top_k[q]) & set(approx_top_k[q]))
    recalls.append(intersection / k)

print(f"Mean Recall@{k}: {np.mean(recalls):.3f}")
print(f"Min  Recall@{k}: {np.min(recalls):.3f}")

:::{.callout-tip}
For a production photo library, run this benchmark on a representative sample of real embeddings before deploying. The default `m=16, ef_construction=64` HNSW parameters typically achieve recall $> 0.95$ for $N < 10^6$; tuning `ef_search` at query time lets you shift the recall–latency tradeoff without rebuilding the index.

:::

## Appendix: Text-to-Image vs. Image-to-Image Search {#sec-search-modes}

CLIP's shared embedding space supports three search modes that differ only in how the query vector is constructed:

**Text-to-image.** Encode the query string with the text encoder: $\mathbf{q} = f_T(\text{query})$. This is the primary search mode exposed by `POST /search/`.

**Image-to-image.** Encode an uploaded image with the image encoder: $\mathbf{q} = f_I(\text{image})$. The `GET /photos/{id}/similar` endpoint is a special case where the uploaded image is already in the database. For a "find photos like this" upload flow, the Flet client sends the image bytes to a new `POST /search/similar-image` endpoint, which calls `embed_photo` and queries pgvector with the result.

**Combined (text + image).** A weighted sum of the two query embeddings:

$$\mathbf{q} = \frac{\alpha \, f_T(\text{text}) + (1 - \alpha) \, f_I(\text{image})}{\|\alpha \, f_T(\text{text}) + (1 - \alpha) \, f_I(\text{image})\|}$$

where $\alpha \in [0, 1]$ controls how much weight to give the text vs. image component. This is the same technique used in DALL-E's *CLIP guidance* and in tools like [Imagen](https://imagen.research.google/). For the photo app, $\alpha = 0.5$ is a sensible default; the Flet UI could expose a slider.

Implementing the combined query vector:

In [ ]:
def combined_query_vec(
    text: str,
    image_bytes: bytes,
    alpha: float = 0.5,
) -> np.ndarray:
    """Return a normalized combined text+image query embedding."""
    tokens = clip.tokenize([text]).to(device)
    img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    img_tensor = preprocess(img).unsqueeze(0).to(device)

    with torch.no_grad():
        t_vec = model.encode_text(tokens)
        i_vec = model.encode_image(img_tensor)
        t_vec = t_vec / t_vec.norm(dim=-1, keepdim=True)
        i_vec = i_vec / i_vec.norm(dim=-1, keepdim=True)

    combined = alpha * t_vec + (1 - alpha) * i_vec
    combined = combined / combined.norm(dim=-1, keepdim=True)
    return combined.squeeze(0).cpu().numpy().astype(np.float32)


# Verify shape with a synthetic image
buf = io.BytesIO()
Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8)).save(buf, format="JPEG")
q = combined_query_vec("night sky", buf.getvalue(), alpha=0.7)
print(f"Combined query shape : {q.shape}")
print(f"L2 norm              : {np.linalg.norm(q):.6f}")

---

■